# Homework: Logistic Regression with Train/Validation Split & Hyperparameter Tuning

In this notebook, we utilize the functions implemented in `ml_fundamentals.py` to:
1. Generate and visualize a 2D binary classification dataset split into training and validation sets[cite: 1, 2].
2. Implement a gradient ascent training loop tracking both training and validation cross-entropy loss[cite: 1, 2].
3. Visualize the learned decision boundary against training and validation data[cite: 2].
4. **Bonus:** Analyze the impact of learning rate ($\eta$) on convergence speed and stability[cite: 1, 2].

In [ ]:
# download the necessary libraries / Installation des libraires nécessaires
!pip install numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
import importlib

# Import student implementations / Importer les implémentations de l'étudiant·e
import ml_fundamentals
importlib.reload(ml_fundamentals)

from ml_fundamentals import (
    logistic_regression_predict,
    cross_entropy_loss,
    logistic_regression_gradient_step
)

# 1. Generate synthetic 2D binary classification dataset / Générer un jeu de données synthétique de classification binaire en 2D
X, y = make_blobs(n_samples=150, centers=2, n_features=2, random_state=42, cluster_std=1.2)

# 2. Split into 80% Train and 20% Validation sets / Séparer en 80% entraînement et 20% validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Visualize dataset split / Visualiser la séparation du jeu de données
plt.figure(figsize=(8, 5))
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], color='crimson', label='Train Class 0', alpha=0.6)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], color='royalblue', label='Train Class 1', alpha=0.6)
plt.scatter(X_val[y_val == 0, 0], X_val[y_val == 0, 1], color='darkred', marker='^', s=80, label='Val Class 0')
plt.scatter(X_val[y_val == 1, 0], X_val[y_val == 1, 1], color='darkblue', marker='^', s=80, label='Val Class 1')

plt.xlabel("Feature 1 ($x_1$)")
plt.ylabel("Feature 2 ($x_2$)")
plt.title("Dataset Overview: Train vs. Validation Split")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

---

## Training Loop Implementation

Weights are initialized uniformly $w^j \sim U(-0.01, 0.01)$ and bias $w_0 = 0.0$[cite: 2]. Gradient updates are performed on the training set while tracking the loss on the validation set

In [ ]:
np.random.seed(42)
N, D = X_train.shape

# Initialize parameters / Initialiser les paramètres
w = np.random.uniform(-0.01, 0.01, size=D)
w0 = 0.0

# Hyperparameters / Hyperparamètres
eta = 0.01
epochs = 800

train_loss_history = []
val_loss_history = []

for epoch in range(epochs):
    # Predict probabilities for train and validation sets / Prédire les probabilités pour les ensembles d'entraînement et de validation
    y_pred_train = logistic_regression_predict(X_train, w, w0)
    y_pred_val = logistic_regression_predict(X_val, w, w0)
    
    # Compute cross-entropy loss / Calculer la perte d'entropie croisée
    loss_tr = cross_entropy_loss(y_train, y_pred_train)
    
    train_loss_history.append(loss_tr)
    
    # Gradient ascent update on training set only / Mise à jour par montée de gradient, uniquement sur l'ensemble d'entraînement
    w, w0 = logistic_regression_gradient_step(X_train, y_train, y_pred_train, w, w0, eta)

print(f"Training Complete!")
print(f"Final Train Loss: {train_loss_history[-1]:.4f}")
print(f"Learned Weights w: {w}")
print(f"Learned Bias w0:   {w0:.4f}")

## Loss Curve Convergence

Plotting train and validation loss curves confirms gradient ascent optimizes loss without overfitting.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_loss_history, color='tab:blue', linewidth=2, label='Train Loss')
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training and Validation Loss Convergence")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

---

## Decision Boundary Visualization

The linear boundary $w_1 x_1 + w_2 x_2 + w_0 = 0$ is plotted alongside data points[cite: 2].

In [ ]:
plt.figure(figsize=(9, 6))

# Scatter plot of samples / Nuage de points des exemples
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], color='crimson', label='Train Class 0', alpha=0.5)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], color='royalblue', label='Train Class 1', alpha=0.5)
plt.scatter(X_val[y_val == 0, 0], X_val[y_val == 0, 1], color='darkred', marker='^', s=80, label='Val Class 0')
plt.scatter(X_val[y_val == 1, 0], X_val[y_val == 1, 1], color='darkblue', marker='^', s=80, label='Val Class 1')

# Calculate decision boundary points / Calculer les points de la frontière de décision
x1_vals = np.array([X[:, 0].min() - 0.5, X[:, 0].max() + 0.5])
x2_vals = -(w[0] * x1_vals + w0) / w[1]

# Plot linear decision boundary / Tracer la frontière de décision linéaire
plt.plot(x1_vals, x2_vals, color='black', linestyle='--', linewidth=2.5, label='Decision Boundary')

plt.xlim(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
plt.ylim(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
plt.xlabel("Feature 1 ($x_1$)")
plt.ylabel("Feature 2 ($x_2$)")
plt.title("Logistic Regression Decision Boundary")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

---

## Bonus: Learning Rate ($\eta$) Sensitivity Analysis

Comparing model convergence across varying learning rates ($\eta \in [0.0001, 0.005, 0.05]$)

In [ ]:
def run_experiment(learning_rate, num_epochs=300):
    np.random.seed(42)
    w_exp = np.random.uniform(-0.01, 0.01, size=D)
    w0_exp = 0.0
    history = []
    
    for _ in range(num_epochs):
        y_pred = logistic_regression_predict(X_train, w_exp, w0_exp)
        loss = cross_entropy_loss(y_train, y_pred)
        history.append(loss)
        w_exp, w0_exp = logistic_regression_gradient_step(X_train, y_train, y_pred, w_exp, w0_exp, learning_rate)
        
    return history

learning_rates = [0.0001, 0.005, 0.05]

plt.figure(figsize=(9, 5))
for lr in learning_rates:
    loss_hist = run_experiment(lr)
    plt.plot(loss_hist, linewidth=2, label=f"eta = {lr}")

plt.xlabel("Epochs")
plt.ylabel("Train Cross-Entropy Loss")
plt.title("Bonus: Impact of Learning Rate (eta) on Loss Optimization")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()